<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/ColabMD_Edu_Protein_NA_Run_CGMD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Mount to GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# @title Create the folder > ColabMD-Edu_Protein-NA
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-NA"

In [3]:
import sys
sys.path.append('/usr/local/lib/python3.13/site-packages')

In [ ]:
# @title Change Directory > ColabMD-Edu_Protein-NA!pip install -q condacolab
%%bash
# Download the latest Miniforge installer
wget -q -O Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"

# Install it silently into Colab's main directory
bash Miniforge3.sh -b -f -p /usr/local

# Clean up the installer script
rm Miniforge3.sh

# Set conda-forge as the default channel
conda config --system --add channels conda-forge
conda config --system --set channel_priority strict

In [5]:
# @title Force remove the hidden configuration pin file causing the glitch
!rm -f /usr/local/conda-meta/pinned

In [ ]:
# @title Install GROMACS via mamba cleanly (T4 GPU)
!mamba install -c conda-forge gromacs=*=*cuda* -y

In [ ]:
# @title Verify it is installed and using AVX2 & CUDA
!gmx --version

In [8]:
# @title Change Directory > ColabMD-Edu_Protein-NA
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA')

In [ ]:
# @title Clone CGMD directory from GitHub
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/MD/CGMD/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d CGMD

**Coarse-Grained MD Simulation**

In [10]:
# @title Change Directory > CGMD
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD')

In [ ]:
# @title Grompp minimization.tpr file
!gmx grompp -f minimization.mdp -c system.gro -p system.top -o minimization.tpr -maxwarn 10

In [ ]:
# @title Run minimization simulation
!gmx mdrun -deffnm minimization -v

In [ ]:
# @title Make index file (Protein_DNA;15 & W_ION;16)
!echo -e "\"Protein\" | \"DNA\"\n\"W\" | \"ION\"\nq" | gmx make_ndx -f minimization.gro -o index.ndx

In [ ]:
# @title Grompp equlibration.tpr file
!gmx grompp -f equilibration.mdp -c minimization.gro -p system.top -o equilibration.tpr -maxwarn 5 -n index.ndx

In [ ]:
# @title Run equilibration simulation
!gmx mdrun -deffnm equilibration -v

In [ ]:
# @title Grompp dynamic.tpr file
!gmx grompp -f dynamic.mdp -c equilibration.gro -p system.top -o dynamic.tpr -maxwarn 5 -n index.ndx

In [ ]:
# @title Run dynamic simulation
!gmx mdrun -deffnm dynamic -v

**Centered and Fitted Trajectories**

In [ ]:
# @title Center the trajectories
!echo -e "Protein_DNA\nSystem" | gmx trjconv -pbc mol -center -ur compact -s dynamic.tpr -n index.ndx -f dynamic.xtc -o dynamic_center.xtc

In [ ]:
# @title Make new index (BB;17)
!echo -e "a BB\nq" | gmx make_ndx -f equilibration.gro -n index.ndx

In [ ]:
# @title Fit the trajectories
!echo -e "BB\nSystem" | gmx trjconv -fit rot+trans -s dynamic.tpr -f dynamic_center.xtc -o dynamic_fitted.xtc -n index.ndx

In [ ]:
# @title Make multiframe pdb
!echo -e "Protein_DNA" | gmx trjconv -s dynamic.tpr -n index.ndx -f dynamic_center.xtc -dt 1000 -o dynamic.pdb

In [ ]:
# @title Zip and download simulation files
import os
import zipfile
from google.colab import files

target_files = [
    "dynamic_fitted.xtc",
    "dynamic.mdp",
    "dynamic.tpr",
    "dynamic.pdb",
    "index.ndx"
]

zip_name = "simulation_results.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in target_files:
        if os.path.exists(file):
            zf.write(file)
            print(f"Added: {file}")
        else:
            print(f"Warning: {file} not found, skipping.")

files.download(zip_name)